# SARIMAX Universal 


## 1. Imports

In [ ]:
from pathlib import Path
from itertools import product
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

## 2. CONFIGURAÇÃO


In [ ]:
CSV_PATH = Path("brazil.csv")

DATE_COL = "date"
TARGET_COL = "goals_brazil"

# Variáveis externas numéricas disponíveis na origem da previsão
EXOG_NUMERIC = [
    "is_home",
    "days_since_last_match",
    "month_sin",
    "month_cos",
    "doy_sin",
    "doy_cos",
    "year",
]

# Variáveis externas categóricas disponíveis na origem
EXOG_CATEGORICAL = [
    "Competition",
]

TEST_SIZE = 0.20
HORIZON = 1

# Altere de acordo com a sazonalidade da base.
SEASONAL_PERIODS = [4, 8, 12]

# Busca
P_VALUES = [0, 1, 2]
D_VALUES = [0, 1]
Q_VALUES = [0, 1, 2]

SEASONAL_P_VALUES = [0, 1]
SEASONAL_D_VALUES = [0, 1]
SEASONAL_Q_VALUES = [0, 1]

MAX_CANDIDATES = 25
RANDOM_STATE = 42

## 3. Adaptador do exemplo `brazil.csv`

Este bloco é **específico da base do Brasil**.  
Quando usar o SARIMAX universal em outra base, substitua esta função pela preparação da nova base.


In [ ]:
def prepare_brazil(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path).copy()

    df["date"] = pd.to_datetime(
        df["Date"],
        format="%d %b %Y",
        errors="coerce",
    )

    df = (
        df.sort_values("date", kind="stable")
        .drop_duplicates(subset=["date"], keep="first")
        .reset_index(drop=True)
    )

    score = df["Score"].astype("string").str.extract(
        r"^\s*(\d+)\s*-\s*(\d+)"
    )

    df["home_goals"] = pd.to_numeric(
        score[0],
        errors="coerce",
    )
    df["away_goals"] = pd.to_numeric(
        score[1],
        errors="coerce",
    )

    df["is_home"] = (
        df["Match"]
        .str.startswith("Brazil v ")
        .astype(int)
    )

    df["goals_brazil"] = np.where(
        df["is_home"].eq(1),
        df["home_goals"],
        df["away_goals"],
    )

    df["days_since_last_match"] = (
        df["date"].diff().dt.days.clip(lower=0)
    )

    df["month"] = df["date"].dt.month
    df["month_sin"] = np.sin(
        2 * np.pi * df["month"] / 12
    )
    df["month_cos"] = np.cos(
        2 * np.pi * df["month"] / 12
    )

    df["day_of_year"] = df["date"].dt.dayofyear
    df["doy_sin"] = np.sin(
        2 * np.pi * df["day_of_year"] / 365.25
    )
    df["doy_cos"] = np.cos(
        2 * np.pi * df["day_of_year"] / 365.25
    )

    df["year"] = df["date"].dt.year

    required = (
        [DATE_COL, TARGET_COL]
        + EXOG_NUMERIC
        + EXOG_CATEGORICAL
    )

    df = df.dropna(subset=required).copy()

    return df.reset_index(drop=True)


df = prepare_brazil(CSV_PATH)

print("Observações:", len(df))
display(df.head())

## 4. Funções universais

In [ ]:
def validate_temporal_data(
    df,
    date_col,
    target_col,
    exog_numeric,
    exog_categorical,
):
    required = (
        [date_col, target_col]
        + list(exog_numeric)
        + list(exog_categorical)
    )

    missing = [
        col
        for col in required
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Colunas ausentes: {missing}"
        )

    out = df.copy()
    out[date_col] = pd.to_datetime(
        out[date_col],
        errors="coerce",
    )

    out = (
        out.dropna(subset=required)
        .sort_values(date_col, kind="stable")
        .reset_index(drop=True)
    )

    return out


def build_exog_preprocessor(
    numeric_cols,
    categorical_cols,
):
    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])

    transformers = []

    if numeric_cols:
        transformers.append(
            (
                "numeric",
                numeric_pipe,
                numeric_cols,
            )
        )

    if categorical_cols:
        transformers.append(
            (
                "categorical",
                categorical_pipe,
                categorical_cols,
            )
        )

    if not transformers:
        return None

    return ColumnTransformer(transformers)


def temporal_split(df, test_size):
    split_idx = int(
        len(df) * (1 - test_size)
    )

    return (
        df.iloc[:split_idx].copy(),
        df.iloc[split_idx:].copy(),
    )

In [ ]:
def build_candidate_space(
    p_values,
    d_values,
    q_values,
    seasonal_p_values,
    seasonal_d_values,
    seasonal_q_values,
    seasonal_periods,
    max_candidates=None,
    random_state=42,
):
    candidates = []

    for p, d, q, P, D, Q, m in product(
        p_values,
        d_values,
        q_values,
        seasonal_p_values,
        seasonal_d_values,
        seasonal_q_values,
        seasonal_periods,
    ):
        if (
            p == 0
            and d == 0
            and q == 0
            and P == 0
            and D == 0
            and Q == 0
        ):
            continue

        candidates.append(
            (
                (p, d, q),
                (P, D, Q, m),
            )
        )

    rng = np.random.default_rng(
        random_state
    )
    rng.shuffle(candidates)

    if max_candidates is not None:
        candidates = candidates[
            :max_candidates
        ]

    return candidates


def search_sarimax(
    y_train,
    X_train,
    candidates,
):
    results = []

    best_bic = np.inf
    best_order = None
    best_seasonal_order = None

    for order, seasonal_order in candidates:
        try:
            fitted = SARIMAX(
                y_train,
                exog=X_train,
                order=order,
                seasonal_order=seasonal_order,
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(
                disp=False,
                maxiter=250,
            )

            results.append({
                "order": order,
                "seasonal_order": seasonal_order,
                "AIC": fitted.aic,
                "BIC": fitted.bic,
            })

            if (
                np.isfinite(fitted.bic)
                and fitted.bic < best_bic
            ):
                best_bic = fitted.bic
                best_order = order
                best_seasonal_order = (
                    seasonal_order
                )

        except Exception:
            continue

    table = pd.DataFrame(results)

    if table.empty:
        raise RuntimeError(
            "Nenhum SARIMAX convergiu."
        )

    table = table.sort_values(
        "BIC"
    ).reset_index(drop=True)

    return (
        best_order,
        best_seasonal_order,
        best_bic,
        table,
    )

In [ ]:
def walk_forward_sarimax(
    y_train,
    X_train,
    y_test,
    X_test,
    dates_test,
    order,
    seasonal_order,
    horizon=1,
):
    if horizon != 1:
        raise NotImplementedError(
            "Esta implementação universal "
            "usa walk-forward 1-step. "
            "Para h>1 é necessário fornecer "
            "as exógenas conhecidas de cada "
            "passo futuro."
        )

    predictions = []

    for i in range(len(y_test)):
        y_history = np.concatenate([
            y_train,
            y_test[:i],
        ])

        if X_train is None:
            X_history = None
            X_next = None
        else:
            X_history = np.vstack([
                X_train,
                X_test[:i],
            ])
            X_next = X_test[i:i+1]

        fitted = SARIMAX(
            y_history,
            exog=X_history,
            order=order,
            seasonal_order=seasonal_order,
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(
            disp=False,
            maxiter=250,
        )

        pred = fitted.get_forecast(
            steps=1,
            exog=X_next,
        ).predicted_mean

        predictions.append(
            float(np.ravel(pred)[0])
        )

    predictions = np.asarray(
        predictions
    )

    result = pd.DataFrame({
        "date": dates_test,
        "real": y_test,
        "prediction": predictions,
        "absolute_error": np.abs(
            y_test - predictions
        ),
    })

    mae = mean_absolute_error(
        y_test,
        predictions,
    )

    return mae, result

## 5. Rodar a pipeline universal

In [ ]:
df_model = validate_temporal_data(
    df=df,
    date_col=DATE_COL,
    target_col=TARGET_COL,
    exog_numeric=EXOG_NUMERIC,
    exog_categorical=EXOG_CATEGORICAL,
)

train, test = temporal_split(
    df_model,
    TEST_SIZE,
)

y_train = train[TARGET_COL].to_numpy(
    dtype=float
)
y_test = test[TARGET_COL].to_numpy(
    dtype=float
)

EXOG_COLS = (
    EXOG_NUMERIC
    + EXOG_CATEGORICAL
)

if EXOG_COLS:
    preprocessor = build_exog_preprocessor(
        EXOG_NUMERIC,
        EXOG_CATEGORICAL,
    )

    X_train = preprocessor.fit_transform(
        train[EXOG_COLS]
    )

    X_test = preprocessor.transform(
        test[EXOG_COLS]
    )
else:
    preprocessor = None
    X_train = None
    X_test = None

print("Treino:", len(train))
print("Teste:", len(test))
print("Exógenas:", EXOG_COLS)

In [ ]:
candidates = build_candidate_space(
    p_values=P_VALUES,
    d_values=D_VALUES,
    q_values=Q_VALUES,
    seasonal_p_values=SEASONAL_P_VALUES,
    seasonal_d_values=SEASONAL_D_VALUES,
    seasonal_q_values=SEASONAL_Q_VALUES,
    seasonal_periods=SEASONAL_PERIODS,
    max_candidates=MAX_CANDIDATES,
    random_state=RANDOM_STATE,
)

(
    best_order,
    best_seasonal_order,
    best_bic,
    search_table,
) = search_sarimax(
    y_train,
    X_train,
    candidates,
)

print("Melhor order:", best_order)
print(
    "Melhor seasonal_order:",
    best_seasonal_order,
)
print("BIC:", round(best_bic, 2))

display(search_table.head(10))

In [ ]:
mae, predictions = walk_forward_sarimax(
    y_train=y_train,
    X_train=X_train,
    y_test=y_test,
    X_test=X_test,
    dates_test=test[DATE_COL].values,
    order=best_order,
    seasonal_order=best_seasonal_order,
    horizon=HORIZON,
)

print(
    "MAE walk-forward:",
    round(mae, 4),
)

display(predictions.head(10))

## 6. Treino final e salvamento

In [ ]:
OUTPUT_DIR = Path("outputs_sarimax_universal")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if EXOG_COLS:
    final_preprocessor = (
        build_exog_preprocessor(
            EXOG_NUMERIC,
            EXOG_CATEGORICAL,
        )
    )

    X_all = (
        final_preprocessor
        .fit_transform(
            df_model[EXOG_COLS]
        )
    )
else:
    final_preprocessor = None
    X_all = None

y_all = df_model[
    TARGET_COL
].to_numpy(dtype=float)

final_model = SARIMAX(
    y_all,
    exog=X_all,
    order=best_order,
    seasonal_order=best_seasonal_order,
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(
    disp=False,
    maxiter=250,
)

predictions.to_csv(
    OUTPUT_DIR
    / "walk_forward_predictions.csv",
    index=False,
)

if final_preprocessor is not None:
    joblib.dump(
        final_preprocessor,
        OUTPUT_DIR
        / "exog_preprocessor.joblib",
    )

# O resultado do statsmodels possui save próprio.
final_model.save(
    OUTPUT_DIR
    / "sarimax_final.pkl"
)

config = {
    "date_col": DATE_COL,
    "target_col": TARGET_COL,
    "exog_numeric": EXOG_NUMERIC,
    "exog_categorical": (
        EXOG_CATEGORICAL
    ),
    "order": list(best_order),
    "seasonal_order": list(
        best_seasonal_order
    ),
    "BIC": float(best_bic),
    "MAE_walk_forward": float(mae),
}

with open(
    OUTPUT_DIR / "config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Arquivos salvos em:",
    OUTPUT_DIR.resolve(),
)

# Como reutilizar em outra base

Troque principalmente:

```python
CSV_PATH = Path("outra_base.csv")
DATE_COL = "nome_da_data"
TARGET_COL = "nome_do_alvo"

EXOG_NUMERIC = [...]
EXOG_CATEGORICAL = [...]

SEASONAL_PERIODS = [...]
```

Depois substitua `prepare_brazil()` pela limpeza da nova base e mantenha as funções universais.

### Escolha de `m`

Exemplos típicos quando a frequência é realmente regular:

- diário com ciclo semanal → `m = 7`
- mensal com ciclo anual → `m = 12`
- trimestral com ciclo anual → `m = 4`
- horário com ciclo diário → `m = 24`

O período deve ser definido de acordo com a frequência e a sazonalidade real da base.
